<div dir="rtl"><h2>05 — שערים צפויים: ההכרעה של M2</h2><p>M2 שנבחר הוא F5: הבסיס ועוד הדגל אין על מה לשחק לשני הצדדים. עמית הכריע ב-27.11.2026 (ה-ש30–ה-ש34). המספרים להלן מתוצרים חתומים; המחברת קוראת אותם ואינה מאמנת.</p></div>

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config/exp005.yaml").exists():
    if root == root.parent:
        raise RuntimeError("Run from the project checkout")
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import json
from collections import Counter

import pandas as pd
import plotly.io as pio
from IPython.display import HTML as _HTML
from IPython.display import display

from ligat.evaluate.goals_experiment import load_goal_experiment
from ligat.viz import goal_training_curves as curves
from ligat.viz import goals_configuration_names as naming
from ligat.viz import goals_decision_display as decision
from ligat.viz import goals_draw_calibration as draw_diagnostics
from ligat.viz import goals_f5_diagnostics as f5_diag
from ligat.viz import goals_f5_review as f5_review
from ligat.viz import goals_m3_dossier as dossier
from ligat.viz import goals_m3_dossier_views as dossier_views
from ligat.viz import goals_notebook as views
from ligat.viz import goals_qa as qa
from ligat.viz import goals_qa_graphs as graphs
from ligat.viz import goals_walk as walk
from ligat.viz import goals_walk_evidence as evidence
from ligat.viz import goals_walk_plots as fixes
from ligat.viz import goals_walk_review as review
from ligat.viz.presentation import display_table


def HTML(content):
    return _HTML(decision.clean(content))


pio.renderers.default = "plotly_mimetype"
spec = load_goal_experiment()
graph_bundle = graphs.bundle(spec)
data = graph_bundle["data"]
report_qa = graph_bundle["report"]
validation_qa = graph_bundle["validation"]
diagnostic = curves.load_learning_diagnostics(spec)
qa_figures = graphs.all_figures(graph_bundle, diagnostic)
qa_figures["candidate_comparison"] = evidence.candidate_figure(graph_bundle)
fixes.apply(qa_figures, graph_bundle)
naming.selectors(qa_figures, graph_bundle)
m3_data = dossier.load_candidates(graph_bundle)
m3_result = draw_diagnostics.augment(dossier.dossier(m3_data), m3_data)
qa_figures.update(dossier_views.figures(m3_result))
explanation_records = []
post_evidence_frames = []


def table(frame, names, *, description=None, **kwargs):
    if description:
        display(HTML('<div dir="rtl"><p>' + description + "</p></div>"))
    frame = frame.copy()
    for field in frame.columns:
        frame[field] = frame[field].map(
            lambda x: naming.configuration_name(x) if isinstance(x, str) else x
        )
    names = dict(names)
    if "lower" in frame and "upper" in frame:
        frame["includes_zero"] = frame.lower.le(0) & frame.upper.ge(0)
        names["includes_zero"] = "כולל 0?"
    precision = dict(kwargs.pop("precision", {}))
    for column in ["lower", "upper", "ci_lower", "ci_upper"]:
        if column in frame:
            precision[column] = 6
    for column in frame.columns:
        if pd.api.types.is_integer_dtype(frame[column]) or column in {
            "n",
            "games",
            "seasons",
            "rows",
            "positive",
            "rank",
        }:
            precision[column] = 0
    kwargs["precision"] = precision
    kwargs.setdefault("values", review.LABELS)
    kwargs.setdefault("show_index", False)
    kwargs.setdefault("unicode_minus", list(frame.select_dtypes(include="number").columns))
    rendered = display_table(frame.rename_axis(None), names, columns=list(frame.columns), **kwargs)
    display(HTML(rendered.content))


def _describe_graph(name, figure, draw):
    if name in {"f5_shap", "f5_home"}:
        before = review.paragraph(f5_diag.explanation(name, f5_result, after=False))
        after = review.paragraph(f5_diag.explanation(name, f5_result, after=True))
    elif name in {"f5_residuals", "f5_rate_calibration", "f5_elo_curve", "f5_seen_unseen"}:
        before = review.paragraph(f5_review.explanation(name, f5_more, after=False))
        after = review.paragraph(f5_review.explanation(name, f5_more, after=True))
    elif name.startswith("m3_"):
        before = dossier_views.explanation(name, m3_result, after=False)
        after = dossier_views.explanation(name, m3_result, after=True)
    elif name == "candidate_comparison":
        before = evidence.candidate_explanation(graph_bundle, after=False)
        after = evidence.candidate_explanation(graph_bundle, after=True)
    else:
        before = fixes.explanation(name, graph_bundle, figure, after=False)
        after = fixes.explanation(name, graph_bundle, figure, after=True)
    before = naming.configuration_name(before)
    after = naming.configuration_name(after)
    display(HTML(before))
    draw()
    display(HTML(after))
    explanation_records.append(
        dict(
            cell=_graph_cell_number,
            graph=name,
            before_length=len(graphs.plain_explanation(before)),
            after_length=len(graphs.plain_explanation(after)),
            before=graphs.plain_explanation(before),
            after=graphs.plain_explanation(after),
        )
    )


def show(figure, name):
    for prefix in ["E3a_", "E2a_", "E3d_"]:
        name = name.removeprefix(prefix)
    _describe_graph(
        name, figure, lambda: naming.figure_labels(review.model_names_figure(figure)).show()
    )


def qa_show(name):
    show(qa_figures[name], name)


def show_html_graph(name, content):
    _describe_graph(name, None, lambda: display(HTML(content)))


display(HTML(walk.input_summary(data)))

In [ ]:
f5_result = f5_diag.diagnostics(data, spec.output)
f5_more = f5_review.review(data, graph_bundle, spec.output, Path(spec.raw["inputs"]["prematch"]))
chapter7_findings = decision.findings_chapter7(f5_result, f5_more)
display(HTML(decision.opening(report_qa, chapter7=chapter7_findings)))

<div dir="rtl"><h2>1 — M2 חוזה שערים לכל קבוצה בבוקר המשחק</h2>
<p>λ הוא ממוצע חיובי של שערים צפויים, לא תוצאה מובטחת ולא הסתברות לניצחון. למשל λ=1.6 אומר 1.6 שערים בממוצע במפגשים חוזרים באותם תנאים; משחק יחיד עדיין יכול להסתיים באפס או בארבעה. M3 מקבל שני λ ובונה מטריצת תוצאות; הסימולציה מתרגמת משחקים לטבלה. התרשים מתאר חוזה נתונים, ללא אומדן או רווח סמך.</p>
</div>

In [ ]:
_graph_cell_number = 6
show_html_graph("chain", walk.chain_html())

<div dir="rtl"><h3>1.1 — המודלים שמשתתפים</h3>
<p><b><a href="#glossary">GLM</a> של Poisson:</b> כל פיצ׳ר מכפיל את קצב השערים בגורם קבוע; למשל ביתיות של פי 1.2 היא דוגמה כללית. מעט פרמטרים נותנים מודל יציב שקל להסביר בכדורגל. EXP-002 הוא סרגל GLM.</p>
<p><b><a href="#glossary">XGBoost</a>:</b> הרבה עצי החלטה קטנים; כל עץ מתקן מעט את טעות הקודמים (boosting). הוא לומד קשרים לא ליניאריים ואינטראקציות, כמו שילוב ביתיות וכוח יריבה, אך במעט נתונים עלול לשנן.</p>
<p><b><a href="#glossary">CatBoost</a>:</b> משפחה דומה של עצים עם טיפול מובנה בקטגוריות, למשל שלב הליגה. משמש להשוואה לשיטת העצים האחרת.</p>
<p><b><a href="#glossary">Maher</a> בסגנון Dixon-Coles:</b> לומד כוח התקפה והגנה לכל קבוצה ויתרון בית. מעדכן את הכוחות לפני כל יום משחקים, כך שניצחונות עבר יכולים לשנות את ציפיית השערים הבאה. הנוסחאות והפרטים בפרק 3.</p></div>

<div dir="rtl"><h2>2.1 — כל משחק תורם שתי שורות, עם יעד משלו</h2>
<p>שורה היא קבוצה במשחק. היעד goals_scored הוא מספר השערים באותו משחק, מתוך matches; goals_for של prematch הוא סכום עבר ואינו היעד. הקלט הוא prematch_v2 הקפוא. ספירת השורות היא מפקד מלא של הנתונים, ולכן אין לה רווח סמך.</p>
</div>

In [ ]:
_graph_cell_number = 9
show(walk.season_games_figure(data), "E3a_season_games")
display(HTML(walk.warmup_html(data)))

<div dir="rtl"><h2>2.2 — בכל קיפול, ההיסטוריה קודמת לעונת המבחן</h2>
<p>Walk-forward מאמן על העבר ובודק על העונה הבאה. עונות החימום אינן משמשות ולידציה; יש 15 קיפולים. מודלי העצים ו-GLM מתאמנים לקיפול. Maher מתעדכן לפני כל יום משחקים, וגם בו משתמשים רק בתוצאות שקדמו לתחילת היום. הצבע מתאר תוכנית קבועה, ולא שיעור שדורש רווח סמך.</p>
</div>

In [ ]:
_graph_cell_number = 11
show(walk.folds_figure(data), "E3a_walk_forward")

<div dir="rtl"><h2>2.3 — ל-XGBoost ול-GLM דעיכה כמעט לא משנה; ל-Maher היא חיונית</h2>
<p>דעיכה היא כמה משחק ישן נספר באימון. ציר x הוא כמה עונות אחורה המשחק נמצא, ציר y הוא המשקל שלו, וכל קו הוא זמן מחצית אחר. זמן מחצית h הוא מספר העונות שאחריהן משחק נספר כחצי.</p>

</div>

$$
w(a;h)=2^{-a/h},\qquad a=s_{validation}-1-s_{training},\qquad w=1\text{ ללא דעיכה}.
$$

<div dir="rtl">
<p>המרחק נמדד מעונת האימון האחרונה, שמשקלה 1; הטווח הוא מעל 0 ועד 1. כשחוזים את 2023/24 עם h=4, העונה 2018/19 נמצאת ארבע עונות לפני עונת האימון האחרונה ולכן משקלה חצי; עם h=1 משקלה 1/16. בלי דעיכה, כל העונות שוות. המשקל אינו מנורמל מחדש. אלו עקומות מתמטיות מדויקות, ללא אי-ודאות מדגמית.</p><p>ב-Maher אותו עיקרון, אך המרחק הוא הפרש הימים חלקי 365.25. הגרף הראשון מציג מרחק בעונות עבור XGBoost ו-GLM; הגרף השני מציג מרחק בימים עבור Maher.</p></div>


In [ ]:
_graph_cell_number = 13
qa_show("season_decay")

In [ ]:
_graph_cell_number = 14
qa_show("daily_decay")

In [ ]:
decay_rows = review.decay_table(graph_bundle)
table(
    decay_rows,
    {
        "model": "מודל",
        "scope": "טווח עונות",
        "half": "זמן מחצית בעונות",
        "nll": "NLL",
        "seasons": "עונות",
    },
    precision={"nll": 6},
    description="ללא דעיכה נמצא בין הטובים בעצים וב-GLM, אך לא ב-Ma"
    "her; בכל טווח מוצגים כל זמני המחצית.",
)
display(HTML(review.decay_conclusion(graph_bundle)))

<div dir="rtl"><h2>3.1 — <a href="#glossary">NLL</a> מודד כמה המודל הופתע ממה שקרה</h2>
<p>Poisson מתאר מספר שערים שלם Y ≥ 0, עם קצב λ &gt; 0. NLL הוא מינוס הלוגריתם הטבעי של ההסתברות שהמודל נתן לתוצאה שנצפתה. ערך נמוך טוב; הטווח הוא מאפס ומעלה.</p></div>

$$
P(Y=y\mid\lambda)=e^{-\lambda}\lambda^y/y!,\qquad NLL=-\ln P(Y=y\mid\lambda).
$$

<div dir="rtl"><p>למשל λ=1.6 נותן לשער אחד הסתברות של כ-0.323, ולכן −ln(0.323)≈1.130. NLL של 1.41 פירושו הסתברות גאומטרית של כ-24.4% למה שקרה; זה ממוצע של הסתברויות בקנה מידה לוגריתמי, ולא אחוז ניחושים נכונים. בכדורגל: מודל טוב נותן יותר משקל לתוצאות שבאמת מתרחשות.</p><p>שוויון ממוצע ושונות נבדק במחברת 02, ולא נמצא פיזור יתר. פרק 7 יבדוק כיול מחדש על המודל. Γ ו-deviance מוסברים במילון. הדוגמה הבאה מתארת משחק אמיתי אך λ=1.6 הוא מספר המחשה בלבד.</p></div>

In [ ]:
from scipy.stats import poisson

figure, example = views.poisson_example(data)
example["game_id"] = example.game_id.astype(str)
table(
    example,
    {
        "game_id": "מזהה משחק",
        "team": "קבוצה",
        "opponent": "יריבה",
        "season": "עונה",
        "observed": "שערים בפועל",
        "lambda_example": "λ להמחשה",
        "nll_example": "NLL להמחשה",
    },
    precision={"nll_example": 4},
    description="זו שורת משחק אמיתית עם קצב Poisson להמחשה "
    "בלבד; מזהה המשחק, הקבוצות והיעד לקוחים "
    "מהדאטה, ולא נוצר כאן יצוא λ של מודל שנבחר.",
)

In [ ]:
_graph_cell_number = 18
show(figure, "poisson_example")

In [ ]:
poisson_rows = pd.DataFrame(
    {
        "goals": range(5),
        "probability": poisson.pmf(range(5), 1.6),
        "nll": -poisson.logpmf(range(5), 1.6),
    }
)
table(
    poisson_rows,
    {"goals": "שערים", "probability": "הסתברות", "nll": "הפתעה NLL"},
    description="הטבלה מחשבת את ההסתברות ואת NLL לכל מספר "
    "שערים בקצב ההמחשה של הגרף. כך אפשר לקשור בין "
    "גובה עמודה לבין מידת ההפתעה במה שנצפה.",
)
display(HTML(review.ruler_html(graph_bundle)))

<div dir="rtl"><h2>3.2 — XGBoost מוסיף עצים שמתקנים את חיזוי השערים</h2>
</div>

$$
\lambda(x)=\exp\left[b+\sum_{t=1}^{T}\eta f_t(x)\right].
$$

<div dir="rtl">
<p>כל עץ לומד תיקון קטן משילוב פיצ׳רים, למשל פער <a href="#glossary">Elo</a> וביתיות. האקספוננט מבטיח λ חיובי. count:poisson הוא יעד האימון, hist היא שיטת הבנייה. CatBoost משתמש גם הוא בעצים ובהפסד Poisson, עם טיפול משותף בקטגוריות. stage מקודד כן/לא ברשת; מזהי קבוצה ויריבה נבדקים בנפרד בסוף ה<a href="#walk-term-0">סולם</a>, עם ערך ״לא מוכרת״ שנקבע מההיסטוריה בלבד.</p>
</div>
<div dir="rtl"><p>דוגמת עץ בעומק 2: קודם ״פער Elo מעל 100?״, אחר כך ״בבית?״. מתקבלים ארבעה קצוות של מצבים כדורגלניים. כל עץ מתקן מעט את לוג קצב השערים. η הוא גודל התיקון ומספר העצים הוא מספר ההזדמנויות לתקן: ביחד זה <a href="#walk-term-6">תקציב למידה</a>. למשל תיקון לוג של 0.1 בקצב 0.05 מוסיף 0.005 ללוג λ, כלומר מכפיל את הציפייה בערך ב-1.005. המונחים count:poisson ו-hist מוסברים בפרק 8.</p></div>

<div dir="rtl"><h2>3.3 — GLM נותן בסיס פשוט, Maher מפריד התקפה והגנה</h2></div>

$$
\lambda_i=\exp(\beta_0+x_i^T\beta),\qquad \lambda_{ij}=\alpha_i\delta_j\gamma^{I(home)}.
$$

<div dir="rtl">
<p>ב-GLM מקדם חיובי מגדיל את קצב השערים, למשל יתרון בית קבוע באותו מערך פיצ׳רים. ב-Maher α היא חוזק ההתקפה, δ היא נטייה לספוג, ו-γ היא מכפלת הביתיות; כולם חיוביים. נדרשת מגבלת זיהוי <bdi dir="ltr" style="white-space:nowrap">Σ log α = 0</bdi>. כאן Dixon–Coles הוא קו בסיס Maher עם דעיכה ועדכון יומי; אין תיקון תלויות לתוצאות נמוכות (ρ).</p>
<p>למשל כוח התקפה פי 1.1 ונטיית יריבה לספוג פי 1.2 מכפילים את קצב השערים פי 1.32 לפני הביתיות. הסימון δ לספיגה נפרד ממקדמי β של ה-GLM. ρ אינו כאן: הוא מתקן צירופי תוצאות של שתי קבוצות, עבודה של M3 שתיבדק במחברת 06. כאן חוזים את מספר שערי כל קבוצה בנפרד.</p>
<p>ה-prior של עולה קטן מ-1 בהתקפה וגדול מ-1 בספיגה, כפי שמחושב בטבלה. זה מונע הנחה שקבוצה שעולה לליגה כבר חזקה כממוצע הליגה. המקור הוא עונות העולות הקפואות, עם עוצמה שקולה לשישה משחקים.</p>
</div>

<details><summary>פרטי ה-<a href="#walk-term-8">prior לעולה</a> — הרחבה מתמטית</summary>
<div dir="rtl"><p>לעולה יש prior קפוא בעוצמה של שישה משחקים, סביב ממוצע ליגת העונה <i>R</i>:</p></div>

$$
\frac{k}{2}\sum_{p\in P}\left[(\log\alpha_p-\overline{\log\alpha}_R-a^*)^2+(\log\delta_p-\overline{\log\delta}_R-b^*)^2\right].
$$

<div dir="rtl">
<p>עוצמת הקנס בפועל היא <bdi dir="ltr" style="white-space:nowrap">k = 6·ȳ<sub>w</sub>·w<sub>0</sub></bdi>: ממוצע שערי העבר המשוקלל מתרגם את ששת המשחקים לעוצמה, ו-w<sub>0</sub> הוא משקל תחילת העונה ביחס לבוקר החיזוי (1 ללא דעיכה). כך גם המידע המוקדם מתיישן עם העונה.</p>
<p>כשיש כמה עולות, נגזרת ממוצע R קושרת את הקנסות. עולה ללא משחקים אינה חייבת להיות בדיוק a*/b*; תנאי האופטימום הוא <bdi dir="ltr" style="white-space:nowrap">d<sub>p</sub> = Σ<sub>q∈P</sub> d<sub>q</sub> / |R|</bdi>. כאן d הוא הסטייה מערך ה-prior ביחס לממוצע R, בנפרד להתקפה ולהגנה. הקנס אינו נכלל ב-NLL הוולידציה.</p>
</div>
</details>


In [ ]:
prior = views.frozen_prior()
table(
    pd.DataFrame(
        [
            dict(
                attack=__import__("math").exp(prior["a_star"]),
                concede=__import__("math").exp(prior["b_star"]),
            )
        ]
    ),
    {"attack": "מכפיל התקפה של עולה", "concede": "מכפיל נטייה לספוג"},
)
table(
    pd.DataFrame([dict(team_seasons=prior["team_seasons"], strength=6)]),
    {"team_seasons": "עונות קבוצה של עולות במקור", "strength": "שווי prior במשחקים"},
)
display(HTML(review.glm_comparison()))

<div dir="rtl"><h2>4 — הרשת קבועה מראש: דיוק מול מורכבות ורגולריזציה</h2>
<p>היפר-פרמטר הוא בחירה שנקבעת מחוץ ללמידת המקדמים: כמה עמוק העץ, כמה תיקונים וכמה קנס. הרשת כוללת ערכים פשוטים ומורכבים, והבחירה לפי ולידציה עתידית. הטבלה נגזרת ישירות מה-YAML. ערך None בזמן מחצית פירושו בלי דעיכה. ברירת מחדל של ספרייה היא מועמד עצמאי מלא. הגרפים בפרקים 4.1–4.3 משווים את הצירופים; ציוני המועמדים הם תוצאות EXP-005 לפני הכרעות עמית.</p>
</div>
<div dir="rtl"><p>שלב A בודק עומק, זהירות, עצים וקצב; שלב B מקבע את הצעת A ובודק דגימה, קנס ודעיכה. ברירת המחדל היא תצורה נפרדת מלאה, ולא ערבוב של ערכים. לדוגמה: עומק 2, 150 עצים וקצב 0.05 מתארים הרבה תיקונים קטנים של מצבים פשוטים. הנימוק הכללי הוא תקציב שמאפשר השוואה בשלוש ספריות בלי חיפוש לא מוגבל; הטבלה הבאה מסבירה למה נבדקו הערכים האלה.</p></div>
<div dir="rtl"><p>תקציב A הוא 5 עומקים × 3 רמות זהירות × 3 מספרי עצים × 2 קצבים, ועוד ברירת מחדל מלאה. ב-B: 3 קנסות × 2 שיעורי דגימה × 4 זמני מחצית. ברירת המחדל של הספרייה שונה מן הרשת; אסור לייחס נקודה בודדת שלה ל״אפקט״ של פרמטר. השילובים והערכים עצמם נקראים מהקונפיגורציה המאושרת.</p></div>
<div dir="rtl"><p>רשת CatBoost כיסתה רק עומק 4 ומעלה ו-1,000 עצים; לכן היא לא השוואה שקולה לרשת XGBoost (עומק מ-2, 150–1,000 עצים). האזור הרדוד נבדק בדיעבד ב-6.2 (P4).</p></div>


In [ ]:
display(
    HTML(
        qa.grid_explanation(load_goal_experiment()).replace(
            "הנימוק הכללי של התקציב ב-§7;", "הרשת נקבעה מראש;"
        )
    )
)
hyper = qa.hyperparameter_guide()
hyper["football"] = hyper.football.str.replace("באותו תקציב", "ברשת המקורית", regex=False)
hyper["reason"] = hyper.parameter.map(
    {
        "max_depth": "2 מאפשר יחסים פשוטים; 8 עמוק מספיק כדי לראות שינון"
        ". הטווח מראה את מחיר המורכבות.",
        "min_child_weight": "1 כמעט בלי דרישת ראיות; 30 דורש תמיכה רחבה יותר לכל חוק.",
        "n_estimators": "150 עד 1,000 בודקים מה קורה כשהתקציב ממשיך אחרי הלמידה הראשונית.",
        "learning_rate": "שני קצבים ברשת מפרידים תיקונים קטנים מהירים יותר; "
        "ברירת הספרייה נבדקת כתצורה מלאה.",
        "half_life": "בלי דעיכה מול זיכרון קצר וארוך; ניסיון חובה של העדפת העבר הקרוב.",
        "library_default": "נקודת הייחוס של הספרייה נבדקת כיחידה שלמה, בלי לערבב רכיבים.",
    }
).fillna("")
table(
    hyper,
    {
        "model": "מודל ושלב",
        "parameter": "פרמטר",
        "values": "ערכים",
        "meaning": "מה הפרמטר עושה",
        "football": "משמעות כדורגלנית",
        "reason": "למה דווקא הערכים האלה",
    },
)
configuration_legend = []
for parameter, column, title, low, high in [
    ("max_depth", "depth", "עומק", "מעט שאלות: כוח ובית", "יותר שילובים וסיכון לשינון"),
    ("n_estimators", "n_estimators", "עצים", "מעט תיקונים", "יותר תיקונים מצטברים"),
    ("learning_rate", "learning_rate", "קצב", "כל תיקון קטן", "כל תיקון משנה יותר"),
    ("min_child_weight", "min_child_weight", "זהירות", "די במעט ראיות", "נדרשות יותר ראיות לחוק"),
    ("reg_lambda", "reg_lambda", "קנס", "חוקים חזקים יותר", "כיווץ החוקים"),
    ("subsample", "subsample", "דגימת משחקים", "פחות מידע בכל עץ", "יותר משחקים בכל עץ"),
    ("half_life", "half_life", "זמן מחצית", "התמקדות בכדורגל האחרון", "יותר משקל לעבר"),
]:
    raw_values = validation_qa.loc[validation_qa.stage_role.isin(["S2-A", "S2-AB"]), column]
    values = [
        f"{100 * float(value):g}%" if column == "subsample" else f"{float(value):g}"
        for value in sorted(raw_values.dropna().unique())
    ]
    if column == "half_life" and raw_values.isna().any():
        values.append("בלי דעיכה")
    configuration_legend.append(
        dict(parameter=parameter, title=title, values=", ".join(values), low=low, high=high)
    )

table(
    pd.DataFrame(configuration_legend),
    {
        "parameter": "פרמטר",
        "title": "בעברית",
        "values": "ערכים ברשת",
        "low": "משמעות ערך נמוך",
        "high": "משמעות ערך גבוה",
    },
)

<div dir="rtl"><h2>4.1 — 15 קיפולים: התאמה לאימון מול יכולת לחזות עונה חדשה</h2>
<p>הבורר בוחר עומק, עצים, קצב למידה וזהירות מתוך 90 תצורות S2-A. כל עקומה מחושבת מיומני full החתומים, וכל נקודת ולידציה מתוצאות full של אותה תצורה. ברירת המחדל תואמת את פרמטרי A של הצעת S2 הקפואה; זו עדיין תוצאות EXP-005 לפני הכרעות עמית.</p>
<p>בכל חישוב יש 15 קיפולים. פס <a href="#g-iqr-ci">IQR</a> מכיל את החצי האמצעי של תוצאות האימון; <a href="#g-iqr-ci">CI95</a> מתאר אי-ודאות בין עונות ולידציה. אימון עשוי להיות משוקלל בדעיכה, ולידציה אינה משוקללת. אין כאן ולידציה לכל איטרציה: הנקודה הסגולה בסוף היא הציון היחיד בוולידציה בלבד.</p></div>


In [ ]:
display(
    HTML(
        review.paragraph(
            "הבורר מחליף עומק, עצים, קצב וזהירות. קבוצת הפיצ׳רים S2-A ואוכלוסי"
            "ית הקיפולים קבועות; התיבה מפרטת את התצורה שנבחרה."
        )
    )
)
_graph_cell_number = 26
qa_show("training_selector")

<div dir="rtl"><h2>4.2 — יותר שינון אינו שיפור מוכח בוולידציה</h2>
<p>הבורר לעיל מאפשר לעבור גם לתקציב גדול יותר. העקומות הן חציון ו-IQR על 15 קיפולים; אין כאן עוד פיילוט של שלושה קיפולים. גרף השפעות הפרמטרים בהמשך כולל רק תצורות רשת. הוא מתאר את הרשת ואת הפיזור בין תצורות, ולא ניסוי סיבתי שמבודד פרמטר אחד.</p></div>


<div dir="rtl"><h2>4.3 — כל ששת תקציבי הלמידה זמינים להשוואה</h2>
<p>בורר עצים וקצב מציג את כל שש הפרוסות; פרוסת ההצעה הקפואה מסומנת ★ ההצעה. בכל מפה השורות הן עומק והעמודות הן זהירות. ירוק טוב ואדום רע; לכל מפה <a href="#walk-term-0">סולם</a> צבע נפרד, ולכן משווים ערכים ולא רק גוונים. במפת תרומת Elo המרכז הוא אפס, וערך שלילי מסומן במינוס.</p>
<p>להגדרה ולדוגמה המחושבת של <a href="#elo-reading">אחוז מתרומת Elo ראו סרגל הקריאה בפרק 3.1</a>.</p></div>


In [ ]:
display(
    HTML(
        review.paragraph(
            "הבורר מחליף עצים וקצב. הצירים מציגים בכל פרוסה את כל ערכי העומק ו"
            "הזהירות; קבוצת הפיצ׳רים ואוכלוסיית הקיפולים קבועות."
        )
    )
)
_graph_cell_number = 29
qa_show("train_vs_validation_heatmap")

In [ ]:
display(
    HTML(
        review.paragraph(
            "הבורר מחליף עצים וקצב. הצירים מציגים בכל פרוסה את כל ערכי העומק ו"
            "הזהירות; קבוצת הפיצ׳רים ואוכלוסיית הקיפולים קבועות."
        )
    )
)
_graph_cell_number = 30
qa_show("hyperparameter_heatmap")

In [ ]:
_graph_cell_number = 31
qa_show("hyperparameter_effects")

In [ ]:
budget = fixes.budget_table(graph_bundle)
table(
    budget,
    {
        "depth": "עומק",
        "trees": "עצים",
        "rate": "קצב",
        "caution": "זהירות",
        "proposal": "ההצעה?",
        "elo": "אחוז מתרומת Elo",
        "lower": "תחתון CI95 מזווג מול R0",
        "upper": "עליון CI95 מזווג מול R0",
        "versus_proposal": "שיפור מזווג מול ההצעה",
        "includes_zero": "כולל 0?",
        "n": "עונות",
    },
    precision={"elo": 2, "lower": 2, "upper": 2, "versus_proposal": 6},
)
display(
    HTML(
        review.paragraph(
            "הטבלה מוצגת בזהירות 30, ונוספה ההצעה בזהירות 1. 0% = רמת רק יתרון בית. "
            "רווח שכולל 0 אומר שאי אפשר לקבוע שהתצורה טובה ממנה; זה קורה לתצורות הכבדות, "
            "שמשננות את העבר ומאבדות את מה ש-Elo נותן. הפס מזווג מול R0, באותו נרמול של האחוזים."
        )
    )
)

<div dir="rtl"><h2>5 — ה<a href="#walk-term-0">סולם</a> בשלוש שורות</h2><p>S1: Elo ובית; אחריו S2: הבסיס המלא.</p><p>כל משפחה נבדקת באותה <a href="#walk-term-4">תצורת כניסה</a> עם ובלי מידע חדש.</p><p>במסלול המקוצר כוונון מחדש מוגבל לתצורות S2 שכבר נשמרו; הוא אינו תרומת המשפחה.</p></div>


<div dir="rtl"><h3>ההפרש המזווג</h3><p>בכל עונה מחסרים את NLL המועמד מציון הייחוס. חיובי פירושו שיפור. הממוצע משוקלל במשחקים; SE מתאר פיזור בין עונות:</p></div>

$$
\bar d=\frac{\sum_s n_s d_s}{\sum_s n_s},\quad SE=\frac{sd(d_s)}{\sqrt{n}},\quad CI_{95\%}=\bar d\pm t_{n-1,.975}SE.
$$


In [ ]:
display(HTML(review.rules_html(graph_bundle)))

In [ ]:
table(views.ladder_plan_table(), {"stage": "שלב", "features": "עמודות", "meaning": "המידע שנבדק"})
feature_details = display_table(
    qa.feature_guide(),
    {"stage": "שלב", "field": "עמודה", "hebrew": "שם בעברית", "purpose": "המידע שנבדק"},
    show_index=False,
)._repr_html_()
display(
    HTML(
        "<details><summary>פירוט הפיצ׳רים בעברית, אחרי המסנן</summary>"
        + feature_details
        + "</details>"
    )
)

<div dir="rtl"><h2>5.1 — מידע דומה מיוצג פעם אחת</h2>
<p>מסנן <a href="#glossary">Pearson</a> מחושב פעם אחת על פיצ׳רים מספריים ובוליאניים בלבד בעונות הפיתוח. מזהים ו-stage אינם מקבלים מספר סדר מלאכותי. r בטווח [−1,1]; כאן r≈0.957, ולכן שני ההפרשים כמעט מספרים אותו סיפור על העונה הקודמת. שתי ההצעות והכרעת עמית מוצגות בלי לייחס למתאם סיבתיות. ספירת הנתונים היא 8,234 שורות תקפות; זו בדיקת חפיפה תיאורית, לא הוכחת תועלת בחיזוי.</p>
</div>
<div dir="rtl"><p>Pearson r מודד קשר ליניארי, בין −1 ל-1. ערך קרוב ל-1 פירושו ששני פיצ׳רים עולים יחד, ולא ששניהם גורמים לשערים. למשל נקודות והפרש שערים של קבוצה חזקה נעים יחד. עונת החימום הראשונה חסרה עבר, ולכן לזוג נתוני עבר יש פחות שורות שלמות. המסנן מחושב על פיצ׳רים בלבד ב-2006/07–2023/24; שערים קודמים משקפים כוח טוב יותר מנקודות, שכבר מקודדות במיקום. הסרת prev_ppm_diff נעשתה לפני D3.</p></div>

In [ ]:
table(
    qa.correlation_evidence(data),
    {
        "pair": "זוג הפיצ׳רים",
        "r": "Pearson r",
        "n": "שורות שלמות",
        "missing": "שורות ללא עבר מלא",
        "position_r": "מתאם נקודות עם מיקום",
        "scope": "טווח",
        "codex": "הצעת Codex",
        "claude": "הצעת Claude",
        "decision": "הכרעת עמית לפני D3",
    },
    precision={"r": 6, "position_r": 6},
    description="הטבלה מתעדת את זוג הפיצ׳רים המתואם, מספר "
    "השורות עם עבר מלא, טווח השנים וההכרעה "
    "המוקדמת. שורות חסרות עבר אינן מתמלאות כאן "
    "באפס כדי להגדיל מדגם.",
)

In [ ]:
_graph_cell_number = 39
qa_show("correlation_scatter")

<div dir="rtl"><h2>6 — התוצאות והכרעת עמית מ-27.11.2026</h2><p>הטבלאות קוראות את ladder_report החתום. בזמן ההרצה סימוני כלל 6 היו זמניים; ההצעות נשארות לפי ה-ש32, ודגלי כלל 10 נסגרו לפי NLL בה-ש34. F5 נבחר כ-M2 בה-ש31; ל-M3 עוברים שלושה לפי ה-ש33. המעמד ההיסטורי נשמר בדוח המקורי, וההכרעה מוצגת מעליו.</p><p>15 הקיפולים נותנים תמונת פיתוח מלאה. תצוגות עמוקים וחמש האחרונות מזוהות בנפרד; טבלאות ההשוואה בעמוקים המשותפים מכסות 13 עונות (2011/12–2023/24).</p></div>

In [ ]:
display(
    HTML(
        review.paragraph(
            "מדדי פרק 6 מבוססים על התוצאות החתומות לכל קיפול."
            " ההשוואה המשותפת משתמשת באותן 13 עונות לכל המודלים."
        )
    )
)

In [ ]:
for scope in ["all", "deep", "last5"]:
    table(
        review.metrics_table(graph_bundle, scope),
        {
            "model": "מודל",
            "nll": "NLL",
            "probability": "הסתברות גאומטרית %",
            "elo": "אחוז מתרומת Elo",
            "mae": "MAE",
            "rmse": "RMSE",
            "better": "באחוז השורות המודל קרוב יותר לתוצאה מ-S2",
            "seasons": "עונות",
        },
        precision={"nll": 6, "mae": 6, "rmse": 6, "elo": 2, "probability": 2},
        description=review.scope_description(scope),
    )
display(
    HTML(
        review.paragraph(
            "F1–F4 לא נכנסו ומוצגות פעם אחת כבסיס מכוונן. "
            "משפחת F6 פוגעת מול F5: "
            '<bdi dir="ltr">d̄=−0.004096, CI95 [−0.007064, −0.001127]</bdi>. '
            "לכן לא נכנסה; תצורת ההמשך שומרת את רשימת F5 ללא המזהים. "
            "הבסיס המכוונן שונה מ-S2 בקנס, בדגימה ובדעיכה של המסלול המקוצר."
        )
    )
)
paired_rows = review.paired_models_table(graph_bundle)

table(
    pd.DataFrame(paired_rows),
    {
        "name": "מודל",
        "d": "שיפור מזווג מול S2",
        "se": "SE",
        "lower": "תחתון CI95",
        "upper": "עליון CI95",
        "n": "עונות",
    },
    precision={k: 6 for k in ["d", "se", "lower", "upper"]},
    intervals={"d": ("lower", "upper")},
)

In [ ]:
entries_qa, f5_seasons = qa.entry_tables(report_qa)
table(
    f5_seasons,
    {"season": "עונה", "d": "d של F5", "games": "משחקים"},
    precision={"d": 6},
    description="כל שורה היא אותה עונת משחקים עם ובלי F5 "
    "בתצורת הכניסה הקבועה. d חיובי הוא שיפור של "
    "המשפחה, ומספר המשחקים קובע את המשקל בממוצע; "
    "אין כאן כוונון נוסף.",
)

In [ ]:
table(
    entries_qa,
    {
        "name": "בדיקה או פירוק F5",
        "d": "שיפור",
        "se": "SE",
        "lower": "תחתון",
        "upper": "עליון",
        "n": "עונות",
        "last5": "שיפור בחמש האחרונות",
        "better": "עדיפות מתוך 5",
        "passes": "כלל 8",
        "entered": "נכנסה",
    },
    intervals={"d": ("lower", "upper")},
    precision={"d": 6, "se": 6, "last5": 6},
    description="אלה בדיקות הבסיס, כניסות המשפחות ופירוק F5. "
    "כל שורה נשענת על הכרעה חתומה, ומפרידה בין "
    "ממוצע שיפור, אי-ודאות, עקביות בחמש האחרונות "
    "ומעמד הכניסה.",
)

In [ ]:
display(HTML('<div id="open-decisions"></div>'))
table(
    review.review_table(graph_bundle),
    {
        "question": "השאלה",
        "proposal": "ההצעה הזמנית",
        "alternative": "החלופה",
        "changes": "מה שונה",
        "gap": "שיפור החלופה מול ההצעה — חיובי טוב",
        "se": "SE",
        "default": "ברירת המחדל",
        "depends": "מה ישתנה בפועל",
    },
    precision={"gap": 6, "se": 6},
    description=(
        "הכרעות זהות אוחדו. כל 11 ההצעות נשארות לפי ה-ש32; הטבלה המלאה מראה כל סימון היסטורי בנפרד."
    ),
)
display(
    HTML(
        "<details><summary>הטבלה המקורית המלאה</summary>"
        + review.render(
            qa.review_evidence(report_qa, validation_qa).assign(
                decision="ההצעה של כלל 4 נשארת (ה-ש32)"
            ),
            {
                "decision": "הכרעת עמית",
                "stage": "שלב",
                "kind": "מועמדת",
                "label": "תצורה",
                "changes": "מה שונה",
                "gap": "פער",
                "se": "SE",
                "n": "עונות",
                "depends": "תלויים",
            },
        )
        + "</details>"
    )
)

In [ ]:
display(HTML(decision.rule10_html(graph_bundle)))

In [ ]:
_graph_cell_number = 47
qa_show("paired_folds")

In [ ]:
_graph_cell_number = 48
qa_show("ladder_progress")

In [ ]:
table(
    evidence.family_ranking(graph_bundle),
    {
        "rank": "מקום",
        "name": "משפחה",
        "meaning": "מה היא מוסיפה",
        "d": "שיפור מזווג",
        "se": "SE",
        "elo": "אחוז מתרומת Elo",
        "better": "חיובי ב-5 האחרונות",
        "passes": "כלל 8",
    },
    precision={"d": 6, "se": 6, "elo": 2},
)
display(HTML(evidence.family_explanation(graph_bundle)))

In [ ]:
_graph_cell_number = 50
qa_show("base_family_forest")

In [ ]:
_graph_cell_number = 51
qa_show("paired_forest")

In [ ]:
_graph_cell_number = 52
qa_show("metric_by_season")

In [ ]:
_graph_cell_number = 53
qa_show("win_rate_by_season")

In [ ]:
_graph_cell_number = 54
qa_show("models_vs_baselines")

In [ ]:
display(HTML(evidence.candidate_tables_html(graph_bundle)))

In [ ]:
_graph_cell_number = 56
qa_show("candidate_comparison")

In [ ]:
_graph_cell_number = 57
qa_show("elo_by_scope")

In [ ]:
_graph_cell_number = 58
qa_show("open_reviews")

In [ ]:
display(HTML(review.learning_description(graph_bundle, diagnostic)))

In [ ]:
_graph_cell_number = 60
qa_show("learning_curves_train_vs_validation")

In [ ]:
table(
    fixes.learning_changes(diagnostic),
    {
        "trees": "מ-150 למספר עצים",
        "median": "חציון השיפור המזווג בוולידציה",
        "better": "קיפולים שהשתפרו",
        "n": "קיפולים באבחון הארוך",
    },
    precision={"median": 6},
)

<div dir="rtl"><h2>6.2 — בדיקות בדיעבד: ראיות היסטוריות והכרעת 27.11.2026</h2>
<p>עונות שכבר נראו; 14 תצורות נבדקו, והמובילה נהנית גם ממזל; לא משנה הכרעה.</p>
<p>אלה 210 אימונים שהושלמו ונשמרו בנפרד: P1 ו-P2 משתמשות בתצורה הקפואה של S2; P3 ו-P4 בודקות אזור שלא כוסה ברשת. אותם שופט, יעדים וקיפולים משמשים גם כאן. synthetic הוא רק תווית השלב במאגר; האימונים רצו על הדאטה האמיתי, בעונות 2006/07–2023/24.</p>
<p>לכל תצורה מוצגות השוואות מזווגות מול S2 ומול GLM סרגל (EXP-002), ובטבלאות נוספות מול M2 שנבחר (F5) S2+F5. כל טווח מחושב בנפרד: כל 15 הקיפולים, 14 העמוקים ו<a href="#walk-term-2">חמש האחרונות</a> (2019/20–2023/24); חיובי פירושו פחות הפתעה בחיזוי. SE ו-CI95 נמדדים בין עונות; אחוז התרומה וההסתברות הגאומטרית הם תיאורים של אותו NLL, ולא ראיה עצמאית נוספת.</p></div>


In [ ]:
_graph_cell_number = 63
qa_show("post_hoc_forest")

In [ ]:
check = "P1"
post_rows, _, _ = graphs.post_hoc_sections(graph_bundle, check)
post_evidence_frames.append(post_rows)
display(HTML(review.post_summary(graph_bundle, check)))

In [ ]:
check = "P2"
post_rows, _, _ = graphs.post_hoc_sections(graph_bundle, check)
post_evidence_frames.append(post_rows)
display(HTML(review.post_summary(graph_bundle, check)))

In [ ]:
check = "P3"
post_rows, _, _ = graphs.post_hoc_sections(graph_bundle, check)
post_evidence_frames.append(post_rows)
display(HTML(review.post_summary(graph_bundle, check)))

In [ ]:
check = "P4"
post_rows, _, _ = graphs.post_hoc_sections(graph_bundle, check)
post_evidence_frames.append(post_rows)
display(HTML(review.post_summary(graph_bundle, check)))

In [ ]:
display(HTML(graphs.post_hoc_chapter_summary(graph_bundle)))

<div dir="rtl"><h2>7 — מה למד מודל השערים שנבחר?</h2>
<p>קודם מוודאים שהעצים המשוחזרים נותנים בדיוק את תחזיות F5 שנבדקו. אחר כך מודדים על שורות ולידציה איך המידע השונה משפיע על התחזית וכמה המודל מעלה את הקצב כשהקבוצה בבית.</p>
<p>בהמשך מוצגים שאריות לפי קבוצה, כיול λ, תחזיות לפי פער Elo, Maher בתחילת העונה ואצל עולות, והשוואה בין משחקים שנראו באימון למשחקים חדשים.</p></div>

<div dir="rtl"><h3>7.1 — שחזור F5</h3><p>העצים שוחזרו עם אותו seed, אותו סדר עמודות ואותם משקלי עונות. בדקנו λ לפי מפתח משחק וקבוצה ואת רצף ה-loss בכל איטרציה. הטבלה קוראת את תוצאות הבדיקה מהמניפסט החתום.</p></div>

In [ ]:
f5_journal = f5_result["journal"].copy()
f5_journal["status"] = f5_journal.bit_identical.map({True: "זהה ביט־לביט"})
table(
    f5_journal[["season_id", "status", "fit_seconds"]],
    {"season_id": "עונת ולידציה", "status": "תוצאת השוואה", "fit_seconds": "שניות אימון"},
    precision={"fit_seconds": 3},
)
display(
    HTML(
        review.paragraph(
            f"זמן השחזור הכולל {f5_result['elapsed_seconds']:.2f} שניות. בבדיקת SHAP "
            f"השגיאה המוחלטת המרבית ב-λ היא {f5_result['max_absolute_error']:.3e} "
            f"והשגיאה היחסית המרבית היא {f5_result['max_relative_error']:.3e}. "
            "הפער הוא עיגול float32 בתוך XGBoost."
        )
    )
)

<div dir="rtl"><h3>7.2 — חשיבות פיצ׳רים</h3><p>SHAP מפרק תחזית על שורות מחוץ לאימון. gain מתאר את השיפור מפיצולים באימון. ערך gain של 20% מציין חמישית מהסכום המנורמל בקיפול. אם פער Elo מקבל ערך גבוה, הפיצולים שמשתמשים בו הפחיתו הרבה מהשגיאה שהעצים התאימו אליה.</p></div>

In [ ]:
_graph_cell_number = 73
show(f5_diag.importance_figure(f5_result), "f5_shap")
flag_fields = {"feature": "דגל פעיל", "rows": "שורות", "mean": "ממוצע", "mean_abs": "ממוצע מוחלט"}
table(f5_result["flags"], flag_fields)
display(
    HTML(
        "<details><summary>gain באחוזים בכל קיפול</summary>"
        + review.render(f5_result["gain"], {"season_id": "עונה", **f5_diag.FEATURE_NAMES})
        + "</details>"
    )
)

<div dir="rtl"><h3>7.3 — יתרון הבית</h3><p>השוואה לשני חיזויים מאותו עץ וליחס השערים בפועל באותה עונה ובאותו שלב. התוצאה מתארת את תגובת המודל להחלפת דגל הבית לבדו.</p></div>

In [ ]:
_graph_cell_number = 75
show(f5_diag.home_figure(f5_result), "f5_home")
home_fields = {
    "stage": "שלב",
    "rows": "שורות",
    "model": "מכפיל המודל",
    "observed": "בנתונים, כולל כל השפעות המגרש",
    "sd": "סטיית תקן בין עונות",
}
table(f5_result["stage"], home_fields, precision={"model": 3, "observed": 3, "sd": 3})
table(f5_result["stage_seasons"], {**home_fields, "season_id": "עונה"})

<div dir="rtl"><h3>7.4 — שאריות לפי קבוצה</h3><p>הבדיקה משתמשת בתחזיות F5 החתומות מחוץ לאימון, בעונות 12–25.</p></div>

In [ ]:
_graph_cell_number = 77
show(f5_review.figure("f5_residuals", f5_more), "f5_residuals")
table(
    f5_more["residuals"].loc[lambda p: p.flag],
    {
        "team": "קבוצה",
        "mean": "שארית",
        "sem": "SE",
        "size": "שורות",
        "side": "מדד",
        "flag": "סימון",
    },
)

<div dir="rtl"><h3>7.5 — כיול λ לפי עשירונים</h3><p>משווים את מספר השערים שהובקעו בממוצע לממוצע התחזית, באותה קבוצה של שורות.</p></div>

In [ ]:
_graph_cell_number = 79
show(f5_review.figure("f5_rate_calibration", f5_more), "f5_rate_calibration")
for _, side in f5_more["calibration_sides"].groupby("side", sort=False):
    table(
        side,
        {
            "side": "מגרש",
            "bin": "קבוצה",
            "predicted": "חזוי",
            "actual": "בפועל",
            "se": "SE",
            "rows": "שורות",
        },
    )

<div dir="rtl"><h3>7.6 — כוח ניטרלי</h3><p>התצוגה הושמטה כי prematch_v2 נשמר לפי משחק ולא לפי תאריך, ובבוקר מחזור יש שורות רק לחלק מהקבוצות (8/4/10 מתוך 14), ותמונת מצב אחידה תיבנה ב-M3/M5.</p></div>

<div dir="rtl"><h3>7.7 — שערים צפויים מול פער Elo</h3><p>מקטעי פער Elo משמשים להשוואה בין התחזיות ותוצאות המשחקים בעונות המשותפות.</p></div>

In [ ]:
_graph_cell_number = 82
show(f5_review.figure("f5_elo_curve", f5_more), "f5_elo_curve")

<div dir="rtl"><h3>7.8 — Maher בתחילת עונה ואצל עולות</h3><p>הטבלאות מחלקות את כל 15 עונות הפיתוח למחזורים 1–5, 6–15 ו-16 ומעלה. הן מראות NLL ושערים בפועל פחות λ, לכל הקבוצות ובנפרד לעולות. Maher מתחיל עם prior בעוצמת שישה משחקים עבור העולה ומעדכן אותו מתוצאות שקדמו ליום המשחק.</p></div>

In [ ]:
for _, population in f5_more["stages"].groupby("population", sort=False):
    table(
        population,
        {
            "population": "אוכלוסייה",
            "model": "מודל",
            "band": "מחזורים",
            "nll": "NLL",
            "residual": "שערים פחות תחזית",
            "rows": "שורות",
        },
    )
display(
    HTML(
        review.paragraph(
            "ב-QA של E4a שיפוע כיול התיקו של Maher היה "
            '<bdi dir="ltr">0.55 [0.22, 0.85]</bdi>, ביחס ל-1 בכיול מושלם. ראו D3.'
        )
    )
)

<div dir="rtl"><h2>8 — הנדסה שומרת על ניסוי עיוור, סיבתי וניתן לחידוש</h2><p>synthetic הוא רק תווית השלב במאגר; האימונים רצו על הדאטה האמיתי, בעונות 2006/07–2023/24.</p>
<p>pilot שומר λ ו-loss אימון בלבד; אין בו יעד ולידציה או NLL, RPS, log loss או Brier. כל יחידה ויומן חתומים, והאינדקס הוא תצוגה נגזרת. חסר או לא עדכני מוצגים במפורש, בלי דריסה או אימון אוטומטי במחברת. חתימת היחידה מכסה 37 תלויות מפורשות וגרסאות ספריות; קוד תצוגה אינו מבטל יחידות. המאמן אינו כותב בכל איטרציה: היומן נאסף בזיכרון ונכתב פעם אחת בסיום.</p><p>Maher נבדק ב-11,600 אימונים יומיים, ללא כשלים. בטבלאות הבאות זמני מפקד הפיילוט: חציון וממוצע ליחידה, ולא אומדן אוכלוסייה. מדידת עובדים היא הרצה אחת לכל מספר, n=60 יחידות קבועות; אין כאן רווח סמך של תזמון חוזר.</p>
</div>
<div dir="rtl"><p>עיוורון: לפני D3 נשמרו חיזויים והנדסה, ללא ציוני מועמדים. סיבתיות: כשחוזים משחק ב-10.12.2023, Maher רשאי להשתמש בתוצאות עד 9.12 בלבד. חידוש: יחידה חתומה קיימת נטענת במקום אימון חוזר. unit_id הוא שם מזהה של חישוב, manifest הוא רשימת חתימות, ו-Parquet הוא פורמט טבלאי דחוס — אין להם משמעות כדורגלנית.</p><p>CatBoost ברירת המחדל נעשה איטי יותר בפיילוט השני כי הקדימות החדשה הפעילה אותו במקביל ליחידות כבדות; זו תחרות על משאבים, לא שינוי בתוצאה. RSS הוא כמה זיכרון ההרצה תפסה בשיא; שיא D3 לא נמדד. יומן האימון נאסף בזיכרון ונכתב פעם אחת ליחידה.</p></div>

In [ ]:
engineering = views.engineering_tables()
table(
    engineering["backend"],
    {
        "timing_group": "קבוצת תזמון",
        "units": "יחידות",
        "fit_mean": "ממוצע D2 בשניות",
        "fit_median": "חציון D2 בשניות",
        "fit_mean_D2b": "ממוצע D2b בשניות",
        "fit_median_D2b": "חציון D2b בשניות",
        "mean_change_percent": "שינוי בממוצע באחוזים",
    },
)
table(
    engineering["wall"],
    {
        "run": "הרצה",
        "seconds": "זמן CLI בשניות",
        "calibration": "מקדם כיול",
        "rss": "שיא RSS כולל ב-MiB",
    },
)
table(
    engineering["bench"],
    {"n_jobs": "תהליכים", "wall_seconds": "זמן קיר בשניות", "units_per_hour": "יחידות בשעה"},
)

In [ ]:
_graph_cell_number = 87
qa_show("worker_bars")

<div dir="rtl"><h3>8 — משחקים שנראו באימון מול משחקי ולידציה</h3><p>שלושה ציונים לאותו עץ בכל קיפול. גבולות האימון נקראים מהיחידה החתומה.</p></div>

In [ ]:
_graph_cell_number = 89
show(f5_review.figure("f5_seen_unseen", f5_more), "f5_seen_unseen")
table(
    f5_more["training"],
    {
        "season_id": "עונה",
        "training": "אימון",
        "weighted": "אימון עם דעיכה",
        "validation": "ולידציה",
        "training_rows": "שורות אימון",
        "validation_rows": "שורות ולידציה",
    },
)

<div dir="rtl"><h3>8.0 — פרטים טכניים ותוצרים שמורים</h3><p>הטבלאות הבאות נועדו לביקורת ולשחזור, ולא לבחירת מודל.</p></div>

In [ ]:
from pathlib import Path

execution = json.loads(Path("data/reports/T-014/D3_execution.json").read_bytes())
table(
    pd.DataFrame(
        [
            dict(
                seconds=execution["wall_seconds"],
                units=len(validation_qa),
                jobs=spec.n_jobs,
                minimum=qa.outcome_diagnostics(validation_qa).loss.min(),
            )
        ]
    ),
    {"seconds": "שניות D3", "units": "יחידות", "jobs": "תהליכים", "minimum": "מינימום ממוצע תצורה"},
)

In [ ]:
prior = views.frozen_prior()
table(
    pd.DataFrame([prior]),
    {
        "a_star": "a* — התקפה יחסית בלוג",
        "b_star": "b* — ספיגה יחסית בלוג",
        "source_sha256": "חתימת מקור",
        "team_seasons": "קבוצות־עונות",
    },
    precision={"a_star": 6, "b_star": 6},
)

In [ ]:
status = views.pilot_status()
table(
    status.groupby("status").size().rename("units").reset_index(),
    {"status": "מצב C5", "units": "יחידות"},
)
table(
    pd.DataFrame(
        [
            dict(
                stage=stage,
                rows=len(curves.training_curves(spec, stage=stage)),
                bytes=(spec.output / stage / "training_curves.parquet").stat().st_size,
            )
            for stage in ["pilot", "full"]
        ]
    ),
    {"stage": "שלב", "rows": "שורות עקומות", "bytes": "גודל בבתים"},
)

In [ ]:
table(
    pd.DataFrame(
        [
            {
                key: diagnostic[key]
                for key in (
                    "configs",
                    "folds",
                    "files",
                    "frozen_artifacts",
                    "frozen_artifacts_unchanged",
                )
            }
        ]
    ),
    {
        "configs": "תצורות אבחון",
        "folds": "קיפולים",
        "files": "קבצים",
        "frozen_artifacts": "מקורות",
        "frozen_artifacts_unchanged": "נשמרו",
    },
)

<div dir="rtl"><h3>8.0.1 — נפח יומנים ותוצאות בדיקות ההנדסה</h3><p>היומנים נכתבים פעם אחת בסוף כל יחידה. נפח הוא מדידה הנדסית, ללא משמעות כדורגלנית. טבלאות החתימות וההיסטוריה נועדו לשחזור בלבד.</p></div>

In [ ]:
journal_sizes = pd.DataFrame(
    [
        dict(
            stage=stage,
            bytes=sum(p.stat().st_size for p in (spec.output / stage).glob("*.trainlog.json.gz")),
            journals=len(list((spec.output / stage).glob("*.trainlog.json.gz"))),
        )
        for stage in ["pilot", "full"]
    ]
)
table(journal_sizes, {"stage": "שלב", "bytes": "בתים דחוסים", "journals": "יומנים"})
outcome_table = qa.outcome_diagnostics(validation_qa)
table(
    pd.DataFrame(
        [
            dict(
                configurations=len(outcome_table),
                minimum=outcome_table.loss.min(),
                flags=int(outcome_table.flag.sum()),
            )
        ]
    ),
    {"configurations": "תצורות", "minimum": "מינימום log loss של H/D/A", "flags": "דגלים מתחת ל-1"},
)

<div dir="rtl"><h2>8.1 — ציון טוב מדי דורש בדיקת דליפה</h2>
<p>log loss של בית/תיקו/חוץ נגזר ממטריצת 6×6: שערים 0–4 ו-5+, ותא שבו שתי הקבוצות מבקיעות חמש ומעלה נחשב תיקו לפי הקירוב של חמש ומעלה. הטווח הוא [0,∞), ופחות טוב יותר. ממוצע תצורה מתחת ל-0.98 עוצר כחשד לדליפה; קיפול בודד רק מסומן. כל תצורה מתחת ל-1.00 מסומנת ל-QA. שלוש התוצאות מסודרות בית/תיקו/חוץ; RPS בטווח [0,1] מתחשב גם בסדר הזה. ״משחקים שנראו מול שלא נראו״ ממתין לבדיקה לאחר בחירת המודל.</p>
</div>
<div dir="rtl"><p>כששתי הקבוצות מבקיעות 5 ומעלה, התא המשותף נספר כתיקו בקירוב מטריצת התוצאות. הסף 0.98 הוא עצירה לחשד לדליפה, ו-1.00 הוא דגל לבדיקה. למשל הצצה לתוצאה ב-Elo הופכת חיזוי לקל באופן מלאכותי; זו הדגמת דליפה ולא מודל מועמד. נתוני האימות בפועל מופיעים בטבלה הבאה.</p></div>

In [ ]:
display(HTML(graphs.historical_leakage_reference()))

In [ ]:
display(HTML(evidence.leakage_html(graph_bundle)))

In [ ]:
_graph_cell_number = 100
qa_show("hda_histogram")

<div dir="rtl"><h2 id="m3-dossier">תיק מועמדים ל-M3</h2>
<p>על סמך התיק הזה הוכרע ב-27.11.2026: ל-M3 עוברים F5, GLM בבסיס ו-Maher (ה-ש33).</p><p>הפרק מתאר מה מקבל שלב הכיול משבעת המועמדים שכבר הוערכו. אלה אותם ערכי λ מהתוצרים החתומים, במדיניות as_evaluated.</p>
<p>כל הקיפולים, העמוקים המשותפים וחמש האחרונות מוצגים בנפרד. ההשוואות משתמשות באותם משחקים, והבדיקות בדיעבד מסומנות בכל תצוגה.</p>
<p>התיק עוסק בתיקו, בתוצאות נמוכות, בכיול, בדמיון ובסוגי המשחקים שבהם התחזיות שונות. הוא מכין ראיות לדיון ואינו בוחר מודל או ממליץ על צירוף.</p><p>ב-QA של E4a (27.11.2026), שיפוע כיול התיקו של Maher היה <bdi dir="ltr">0.55 [0.22, 0.85]</bdi> מול 1 בכיול מושלם. ראו D3.</p></div>

In [ ]:
_graph_cell_number = 102
qa_show("m3_draw")
display(
    HTML(
        "<details><summary>D1 — כל הטווחים והספירות</summary>"
        + dossier_views.table(m3_result["draw"])
        + "</details>"
    )
)

<div dir="rtl"><h3>D2 — האם התחזית נותנת מספיק משקל לתוצאות נמוכות?</h3><p>הטבלה מציגה שכיחות חזויה ונצפית של 0-0, 1-0, 0-1 ו-1-1 בעמוקים המשותפים. יחס חזוי/נצפה של 1 מציין התאמה ממוצעת. בבלוק המלא מופיעים כל הטווחים, וגם סך שערים 0, 1, 2, 3, 4 ו-5+; אין חיתוך של זנב ההסתברות.</p></div>

In [ ]:
display(HTML(dossier_views.section_table(m3_result, "D2")))
display(
    HTML(
        review.paragraph(
            "בכדורגל, חסר ב-0-0 או ב-1-1 יכול להשאיר לשלב הכיול תיקון שונה מחסר כללי בתיקו. "
            "היחסים מתארים את שכיחות התוצאות ולא את סדר העדיפות של המודלים; יחס חסר או עודף "
            "אינו מסביר לבדו את מנגנון הטעות, ויש לקרוא אותו יחד עם מספר המשחקים."
        )
    )
)

In [ ]:
_graph_cell_number = 105
qa_show("m3_calibration")
display(
    HTML(
        "<details><summary>D3 — עשר קבוצות הסתברות, כל הטווחים והספירות</summary>"
        + dossier_views.calibration_table(m3_result, "fixed")
        + "</details>"
    )
)
display(
    HTML(
        "<details><summary>תיקו — קבוצות לפי מספר משחקים</summary>"
        + dossier_views.calibration_table(m3_result, "deciles")
        + "</details>"
    )
)

In [ ]:
_graph_cell_number = 106
qa_show("m3_diversity")

<div dir="rtl"><h3>D5 — באילו משחקים התחזיות שונות?</h3><p>לכל מועמד מוצגים NLL ו-RPS בשלישי האיזון של R1, בשלבי העונה ובמשחקים עם דגל nothing_to_play לאחת הקבוצות. השיפור המזווג הוא ציון F5 פחות ציון המועמד על אותם משחקים. SE וה-CI95 מחושבים בין עונות; גם מספר העונות והמשחקים מוצג. פלייאוף אמצעי היסטורי נשמר כקבוצה נפרדת בטווח המלא.</p></div>

In [ ]:
display(HTML(dossier_views.section_table(m3_result, "D5")))
display(
    HTML(
        review.paragraph(
            "פער בקבוצת משחקים אחת יכול להיות קשור לסוג המידע: כוח יחסי, שלב העונה או מוטיבציה. "
            "אלה תיאורים על קבוצות שכבר נצפו, ולא הוכחה שהדגל או שלב העונה גורמים לטעות. "
            "טווח שכולל אפס אינו מפריד בבירור בין המועמד ל-F5 באותה קבוצה."
        )
    )
)

<div dir="rtl"><h3>D6 — סטטיסטי הפיזור של פירסון</h3><p>לכל שורת קבוצה-משחק מחשבים <bdi dir="ltr">(Y − λ)² / λ</bdi>, ומציגים את הממוצע בכל טווח. SE מחושב מפיזור הממוצעים העונתיים, ומספר השורות הוא פעמיים מספר המשחקים.</p><p>תחת תחזית Poisson נכונה, התוחלת של המדד היא 1. ערך מעל 1 יכול לנבוע מפיזור יתר אמיתי, וגם מטעויות של המודל ב-λ, והמדד לא מפריד ביניהן. לכן הוא בר השוואה בין מועמדים על אותם משחקים, ולא הוכחה לפיזור יתר.</p></div>

In [ ]:
display(HTML(dossier_views.section_table(m3_result, "D6")))
display(
    HTML(
        review.paragraph(
            "בקבוצות כדורגל שונות קצב השערים שונה; פיזור גדול של התוצאות יכול לשקף גם תחזית ממוצע "
            "שלא מתאימה למשחק. מדד פירסון משווה את השגיאות לקצב של אותו משחק, אך אינו מספק "
            "לבדו הכרעה על התפלגות חלופית או על מועמד שיעבור ל-M3."
        )
    )
)

<div dir="rtl"><h3>סיכום ניטרלי של תיק המועמדים</h3><p>כל המדדים האבחוניים כאן הם בעמוקים המשותפים. עמודת NLL חמש אחרונות משתמשת בחמש העונות האחרונות. סדר השורות קבוע לפי רשימת המועמדים, ללא דירוג וללא צבעי טוב/רע. "פיזור" הוא המדד המותנה של פירסון; המתאם הגבוה מתעלם מהאלכסון העצמי.</p></div>

In [ ]:
display(HTML(dossier_views.section_table(m3_result, "summary")))

<div dir="rtl"><h2>9 — שכבת λ החתומה היא הקלט ל-M3</h2><p>שכבת m2_lambda_v1 כוללת F5, GLM בבסיס ו-Maher. מדיניות as_evaluated שומרת את היסטוריית ההערכה המקורית. Maher יומי רשאי להשתמש בתוצאות מוקדמות מאותה עונה שקדמו ל-as_of. train_end_season מציין את העונה השלמה האחרונה, ו-train_end_date את תאריך המשחק האחרון באימון.</p><p>חוזה עשר העמודות נשמר. folds.parquet מוסיף היסטוריית אימון ועומק חתומים לכל עונה. עונות 11–12 דלות ב-GLM בבסיס ועונה 11 דלה ב-F5 וב-Maher. הטבלה מאמתת את ה-NLL של פרק 6 בכל אחד משלושת הטווחים, ומציגה את עומק האימון של כל מודל בנפרד.</p></div>

In [ ]:
from ligat.evaluate.goals_lambda_bundle import load_bundle_summary

lambda_bundle = load_bundle_summary(spec, data)
table(
    pd.DataFrame(
        {
            "name": record["name"],
            "role": record["role"],
            "rows": record["rows"],
            "games": record["games"],
            "seasons": record["seasons"],
            "deep": record["deep_seasons"],
            "sha": record["lambda_sha256"][:12],
            "fold_sha": record["folds_sha256"][:12],
            "all": record["nll"]["all"],
            "deep_nll": record["nll"]["deep"],
            "last5": record["nll"]["last5"],
            "status": "זהה לפרק 6",
        }
        for record in lambda_bundle["models"].values()
    ),
    {
        "name": "מודל",
        "role": "תפקיד",
        "rows": "שורות",
        "games": "משחקים",
        "seasons": "עונות",
        "deep": "עמוקות",
        "sha": "חתימת λ",
        "fold_sha": "חתימת קיפולים",
        "all": "NLL כלל העונות",
        "deep_nll": "NLL עמוקות",
        "last5": "NLL חמש אחרונות",
        "status": "אימות",
    },
    precision={"all": 6, "deep_nll": 6, "last5": 6},
)

In [ ]:
schema = pd.concat(
    [
        views.lambda_schema_table(),
        pd.DataFrame(
            [
                dict(
                    column="deep_fold",
                    meaning="בקובץ folds.parquet נלווה חתום: האם יש לפחות ארבע עונות אימון",
                )
            ]
        ),
    ],
    ignore_index=True,
)
schema["example"] = schema.column.map(
    {
        "game_id": "משחק המחשה",
        "team_id": "קבוצת המחשה",
        "season_id": "2023/24 (25)",
        "as_of": "10.12.2023",
        "lambda": "1.6 — המחשה",
        "train_end_season": "2022/23",
        "train_end_date": "9.12.2023 ב-Maher",
        "trained_on_current_season": "כן ב-Maher; לא ב-XGBoost",
        "config_id": "חתימת התצורה",
        "prematch_version": "prematch_v2",
        "deep_fold": "כן ב-2023/2024; לא ב-2009/2010",
    }
)
table(schema, {"column": "עמודה", "meaning": "משמעות וחוזה", "example": "דוגמה להמחשה בלבד"})
illustration = pd.DataFrame(
    [
        dict(
            model="Maher",
            as_of="10.12.2023",
            train_end_date="9.12.2023",
            trained_on_current_season=True,
            train_end_season="2022/23",
        ),
        dict(
            model="XGBoost",
            as_of="10.12.2023",
            train_end_date="סוף עונת 2022/23",
            trained_on_current_season=False,
            train_end_season="2022/23",
        ),
    ]
)
table(
    illustration,
    {
        "model": "מודל — המחשת חוזה בלבד",
        "as_of": "בוקר החיזוי",
        "train_end_date": "גבול האימון",
        "trained_on_current_season": "העונה הנוכחית נכללה",
        "train_end_season": "עונה שלמה אחרונה",
    },
)

In [ ]:
display(HTML(decision.final_summary(graph_bundle, m3_result, chapter7=chapter7_findings)))

In [ ]:
display(
    HTML(
        review.expanded_glossary(
            '<div dir="rtl" id="glossary"><h2>10 — מילון</h2><t'
            "able><thead><tr><th>מושג</th><th>מה זה</th><th>דוג"
            "מה</th><th>משמעות בכדורגל</th></tr></thead><tbody>"
            "<tr><td>Elo</td><td>כוח יחסי המתעדכן מתוצאות עבר</"
            "td><td>פער 100 נקודות</td><td>פערי כוח בין יריבות<"
            "/td></tr><tr><td>GLM</td><td>מודל לוג-ליניארי לקצב"
            " שערים</td><td>ביתיות פי 1.2 — המחשה</td><td>השפעה"
            " קבועה על קצב</td></tr><tr><td>XGBoost</td><td>עצי"
            "ם מתקנים זה את זה</td><td>עץ עומק 2 עם ארבעה קצוות"
            "</td><td>שילובי מצבים</td></tr><tr><td>CatBoost</t"
            "d><td>עצים עם טיפול בקטגוריות</td><td>שלב הליגה הו"
            "א קטגוריה</td><td>השוואת שיטה</td></tr><tr><td>Mah"
            "er</td><td>כוח התקפה וספיגה לכל קבוצה</td><td>התקפ"
            "ה פי 1.1 — המחשה</td><td>כוח קבוצה נפרד לכל צד</td"
            "></tr><tr><td>d̄</td><td>ממוצע משוקלל של הפרשי NLL"
            " מזווגים</td><td>1.42−1.41=0.01</td><td>חיובי פירו"
            'שו חיזוי טוב יותר</td></tr><tr><td><a href="#g-elo'
            '-share">אחוז מתרומת Elo</a></td><td>100 כפול השיפו'
            "ר מעל R0 חלקי שיפור R1 מעל R0</td><td>100% שווה לס"
            "רגל Elo</td><td>כמה מהמידע על הכוח הוחזר</td></tr>"
            "<tr><td>הסתברות גאומטרית</td><td>exp של מינוס NLL "
            "הממוצע</td><td>NLL=1.41 נותן כ-24.4%</td><td>משקל "
            "לתוצאות שבאמת קרו</td></tr><tr><td>RPS</td><td>שגי"
            "אה ריבועית של הסתברויות מצטברות</td><td>0 הוא חיזו"
            "י מושלם</td><td>דירוג תחזית בית/תיקו/חוץ</td></tr>"
            "<tr><td>log loss</td><td>מינוס לוג ההסתברות לתוצאה"
            " שנצפתה</td><td>הסתברות חצי נותנת כ-0.693</td><td>"
            "קנס על ביטחון מוטעה</td></tr><tr><td>Pearson r</td"
            "><td>קשר ליניארי בין −1 ל-1</td><td>r קרוב ל-1 מצב"
            "יע על תנועה יחד</td><td>נקודות והפרש שערים של קבוצ"
            "ה חזקה</td></tr><tr><td>פיזור יתר</td><td>שונות גד"
            "ולה מהממוצע במודל ספירה</td><td>ממוצע 1 ושונות 2 —"
            " המחשה</td><td>יותר משחקי קצה מ-Poisson</td></tr><"
            "tr><td>IQR</td><td>החצי האמצעי של התוצאות</td><td>"
            "1,2,3 נותנים רבעונים 1.5 ו-2.5</td><td>פיזור בין ע"
            "ונות</td></tr><tr><td>SE</td><td>שגיאת תקן של ממוצ"
            "ע בין עונות</td><td>פער קטן מ-SE אינו מובחן ברעש</"
            "td><td>יציבות בין עונות</td></tr><tr><td>כלל 4</td"
            "><td>הצעה פשוטה בתוך SE מהמובילה</td><td>עומק קטן "
            "קודם</td><td>שבירת תיקו נשארת לעמית</td></tr><tr><"
            "td>כלל 6</td><td>מובילות עמוקות ואחרונות שונות</td"
            "><td>שתי תצורות זמן מחצית שונות</td><td>בדיקת שינו"
            "י בליגה</td></tr><tr><td>כלל 8</td><td>שער כניסת מ"
            "ידע עם שלושה תנאים</td><td>F5 בטבלה המחושבת</td><t"
            "d>האם מידע חדש מוסיף</td></tr><tr><td>כלל 10</td><"
            "td>אי-הסכמה בין NLL ל-RPS</td><td>שלושה דגלים בדוח"
            " D3</td><td>רגישות לדרך מדידת איכות</td></tr><tr><"
            "td>walk-forward</td><td>אימון בעבר ובדיקה בעונה הב"
            "אה</td><td>2006/07–2008/09 לפני 2009/10</td><td>מד"
            "מה חיזוי בזמן אמת</td></tr><tr><td>early stopping<"
            "/td><td>עצירה לפי עונת ולידציה</td><td>אינה מופעלת"
            " בניסוי הזה</td><td>מונעת בחירת עצים בדיעבד</td></"
            "tr><tr><td>שלבים A ו-B</td><td>מבנה העץ ואחריו ריס"
            "ון ודעיכה</td><td>עומק קודם, זמן מחצית אחר כך</td>"
            "<td>אין משמעות כדורגלנית ישירה</td></tr><tr><td>המ"
            "סלול המקוצר</td><td>שימוש בחמש תצורות S2 בהמשך</td"
            "><td>אותן תצורות למשפחה חדשה</td><td>תקציב חיפוש מ"
            "וגבל מראש</td></tr><tr><td>הכרעה פתוחה</td><td>בחי"
            "רה שדורשת החלטת עמית</td><td>מובילות שונות בכלל 6<"
            "/td><td>אינה החלטה אוטומטית</td></tr><tr><td>as_of"
            "</td><td>גבול המידע בבוקר המשחק</td><td>משחק ב-10."
            "12 משתמש בעבר בלבד</td><td>מונע הצצה לעתיד</td></t"
            "r><tr><td>D3</td><td>הרצת הסולם המלאה</td><td>תוצא"
            "ות חתומות בפרק 6</td><td>ניסוי הפיתוח</td></tr><tr"
            "><td>Γ ו-deviance</td><td>Γ מכלילה עצרת; deviance "
            "מסיר קבוע של היעד</td><td>Γ(y+1)=y!; אין להשוות de"
            "viance ל-NLL גולמי</td><td>כלי חישוב, לא כוח קבוצה"
            "</td></tr><tr><td>λ</td><td>ממוצע שערים חיובי</td>"
            "<td>1.6 שערים צפויים</td><td>אינו תוצאה מובטחת</td"
            "></tr><tr><td>NLL</td><td>מינוס לוג ההסתברות למה ש"
            "קרה</td><td>−ln(0.323)≈1.130</td><td>הפתעה נמוכה פ"
            "ירושה חיזוי טוב</td></tr><tr><td>Γ</td><td>הכללה ש"
            "ל עצרת</td><td>Γ(3)=2!</td><td>קבוע חישובי של מספר"
            " השערים</td></tr><tr><td>deviance</td><td>הפסד Poi"
            "sson שמחסיר קבוע תלוי יעד</td><td>2[y ln(y/λ)−(y−λ"
            ")]</td><td>אין להשוות ישירות ל-NLL גולמי</td></tr>"
            "<tr><td>RSS</td><td>כמה זיכרון ההרצה תפסה בשיא</td"
            "><td>נמדד בפיילוט, לא נמדד ב-D3</td><td>אין משמעות"
            " כדורגלנית</td></tr><tr><td>QA</td><td>בדיקת איכות"
            " עצמאית</td><td>Claude בודק קוד ומספרים</td><td>מו"
            "נע הכרעות על טעות תצוגה</td></tr><tr><td>unit_id, "
            "manifest, Parquet</td><td>מזהה חישוב, רשימת חתימות"
            " וטבלה דחוסה</td><td>יחידה קיימת נטענת בלי אימון</"
            "td><td>אין משמעות כדורגלנית</td></tr></tbody></tab"
            "le></div>",
            graph_bundle,
        )
    )
)

In [ ]:
counts = Counter(row["cell"] for row in explanation_records)
for row in explanation_records:
    row["multiple_graphs_in_cell"] = counts[row["cell"]] > 1
assert all(
    not row["multiple_graphs_in_cell"] and min(row["before_length"], row["after_length"]) >= 250
    for row in explanation_records
)
Path("data/reports/T-014/E4d_explanations.json").write_text(
    json.dumps(explanation_records, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.concat(post_evidence_frames, ignore_index=True).to_json(
    "data/reports/T-014/E3f_posthoc_scope_evidence.json", orient="records", indent=2
)